# 01 Data selection

This notebook applies the intensity-based spectrum selection procedure to the continuous broadband illumination spectra.

The objective is to retain spectra recorded in the presence of sample material while suppressing background-dominated measurements acquired between sample compartments during continuous translation of the sample holder.

## Project setup

The following cells configure the project root directory and import the required analysis modules.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
from src.cache import load_or_build_spectral_dataset
from src.data_selection import (
    select_sample_related_spectra,
    summarise_selection,
)
from src.loaders import (
    SpectralDataset,
    CONTINUOUS_CSV,
    CONTINUOUS_DIR,
    SAMPLES_CSV,
    load_continuous_spectra,
)
from src.plotting import plot_number_spectra_per_sample

## Selection parameters

The spectrum selection procedure is based on the total spectral intensity of each recorded spectrum.

Spectra are retained if their total intensity exceeds a threshold defined as a multiplicative factor of the median total intensity within the corresponding measurement sequence.

In [ ]:
THRESHOLD_FACTOR = 1.05
FORCE_REBUILD_RAW = False
FORCE_REBUILD_SELECTED = False

## Load continuous spectra

The continuous measurement dataset is loaded from the repository structure.

To reduce loading times during iterative analysis, cached spectral datasets are used whenever possible. The cache is automatically rebuilt if source files or analysis scripts have changed.

In [ ]:
continuous = load_or_build_spectral_dataset(
    name="continuous_spectra_raw",
    builder=load_continuous_spectra,
    source_paths=[
        CONTINUOUS_DIR,
        CONTINUOUS_CSV,
        SAMPLES_CSV,
        PROJECT_ROOT / "src" / "loaders.py",
    ],
    force_rebuild=FORCE_REBUILD_RAW,
)

## Apply spectrum selection

For each spectrum, the total intensity is calculated as the sum over all wavelength channels.

Spectra are retained if:

$$
S_\mathrm{total} \geq f \cdot \tilde{S}_\mathrm{total}
$$

where:

- $S_\mathrm{total}$ is the total intensity of the spectrum,
- $\tilde{S}_\mathrm{total}$ is the median total intensity of the corresponding measurement sequence,
- $f$ is the threshold factor.

In [ ]:
selected_with_background = load_or_build_spectral_dataset(
    name=f"continuous_spectra_selected_{str(THRESHOLD_FACTOR).replace('.', 'p')}",
    builder=lambda: select_sample_related_spectra(
        continuous,
        threshold_factor=THRESHOLD_FACTOR,
    ),
    source_paths=[
        CONTINUOUS_DIR,
        CONTINUOUS_CSV,
        SAMPLES_CSV,
        PROJECT_ROOT / "src" / "loaders.py",
        PROJECT_ROOT / "src" / "data_selection.py",
    ],
    force_rebuild=FORCE_REBUILD_SELECTED,
)

In [ ]:
sample_mask = (
    selected_with_background
    .metadata["is_sample_related"]
    .to_numpy()
)

selected_dataset = SpectralDataset(
    wavelengths=selected_with_background.wavelengths,
    intensities=selected_with_background.intensities[sample_mask],
    metadata=selected_with_background.metadata.loc[sample_mask].reset_index(drop=True),
)

## Selection summary

The following summary reports the number of retained spectra after applying the intensity-based selection criterion.

In [ ]:
summarise_selection(
    original_dataset=continuous,
    selected_dataset=selected_dataset,
)

## Selected spectra per sample class

The following interactive bar plot visualises the number of retained spectra for each sample class after data selection.

In [ ]:
display_order = [
    "Test tube",
    "Microscope slide",
    "Wood",
    "Sand",
    "PE",
    "PE-HD",
    "PE-HD (blue)",
    "PP",
    "PP (green)",
    "PP (orange)",
    "PP (pink)",
    "PP (yellow)",
    "PS",
    "PS (blue)",
    "PS (red)",
    "PS (yellow)",
    "PET",
    "PVC",
]

fig = plot_number_spectra_per_sample(
    selected_dataset,
    display_order=display_order,
)

fig.show()

## Notes

The selected spectra generated in this notebook form the basis for all subsequent preprocessing, dimensionality reduction, and classification analyses.